# Project 01 — Data collection and first profiling

## First-Time Buyer Affordability Pressure by Area

This notebook records the first data collection and profiling stage for the project. The source review establishes whether the official affordability workbook is suitable for analysis before findings are produced.

The profiling stage covers workbook structure, candidate data tables, missing values, data types and duplicate records. These checks support reproducibility and reduce the risk of building analysis on the wrong table or undocumented assumptions.


## Source file

The raw workbook is stored in `data/raw/` using the source link recorded in `data/raw/source_links.md`. Raw files remain unchanged; cleaned outputs are saved separately.


In [ ]:
from pathlib import Path
import re

import pandas as pd
import numpy as np


In [ ]:
PROJECT_ROOT = Path("..").resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
CLEANED_DIR = PROJECT_ROOT / "data" / "cleaned"
RAW_FILE = RAW_DIR / "aff1ratioofhousepricetoworkplacebasedearnings.xlsx"

CLEANED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Expected raw file: {RAW_FILE}")
print(f"Raw file available: {RAW_FILE.exists()}")


## Workbook inspection

ONS workbooks often include cover sheets, notes and metadata. The workbook structure is inspected before the data table is selected.


In [ ]:
excel_file = pd.ExcelFile(RAW_FILE)

sheet_overview = pd.DataFrame({
    "sheet_number": range(1, len(excel_file.sheet_names) + 1),
    "sheet_name": excel_file.sheet_names,
})

sheet_overview


In [ ]:
for sheet_name in excel_file.sheet_names:
    print("\n" + "=" * 80)
    print(f"Sheet: {sheet_name}")
    print("=" * 80)
    preview = pd.read_excel(RAW_FILE, sheet_name=sheet_name, header=None, nrows=12)
    display(preview)


## Candidate data table detection

The scan below searches the top of each sheet for terms likely to appear in the affordability table. It acts as a profiling aid rather than a substitute for analytical judgement.


In [ ]:
def normalise_text(value):
    if pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value)).strip().lower()

search_terms = ["lower quartile", "median", "earnings", "ratio", "area", "local authority"]
sheet_matches = []
header_candidates = []

for sheet_name in excel_file.sheet_names:
    preview = pd.read_excel(RAW_FILE, sheet_name=sheet_name, header=None, nrows=40, dtype=str)
    sheet_text = " ".join(preview.fillna("").astype(str).to_numpy().ravel()).lower()
    matched_terms = [term for term in search_terms if term in sheet_text]
    sheet_matches.append({
        "sheet_name": sheet_name,
        "matched_terms": ", ".join(matched_terms),
        "match_count": len(matched_terms),
    })

    for row_number, row in preview.iterrows():
        row_text = " | ".join(normalise_text(value) for value in row.tolist() if normalise_text(value))
        score = sum(term in row_text for term in ["area", "code", "year", "median", "lower", "quartile", "earnings", "ratio"])
        if score >= 3:
            header_candidates.append({
                "sheet_name": sheet_name,
                "zero_based_row": row_number,
                "excel_row": row_number + 1,
                "score": score,
                "row_text": row_text[:300],
            })

sheet_match_summary = pd.DataFrame(sheet_matches).sort_values(["match_count", "sheet_name"], ascending=[False, True])
header_candidates_df = pd.DataFrame(header_candidates).sort_values(["score", "sheet_name", "excel_row"], ascending=[False, True, True])

display(sheet_match_summary)
display(header_candidates_df.head(20))


## Initial data profile

The highest-scoring candidate is loaded as a structured first pass. The selected sheet and header row remain visible for review before the cleaning stage.


In [ ]:
if not header_candidates_df.empty:
    best_candidate = header_candidates_df.iloc[0]
    DATA_SHEET = best_candidate["sheet_name"]
    HEADER_ROW = int(best_candidate["zero_based_row"])
else:
    DATA_SHEET = excel_file.sheet_names[0]
    HEADER_ROW = 0

print(f"Selected sheet: {DATA_SHEET}")
print(f"Selected header row: {HEADER_ROW} zero-based, Excel row {HEADER_ROW + 1}")

raw_df = pd.read_excel(RAW_FILE, sheet_name=DATA_SHEET, header=HEADER_ROW)
print(f"Rows: {raw_df.shape[0]:,}")
print(f"Columns: {raw_df.shape[1]:,}")
raw_df.head()


In [ ]:
column_profile = pd.DataFrame({
    "column_name": raw_df.columns.astype(str),
    "dtype": [str(dtype) for dtype in raw_df.dtypes],
    "non_null_count": raw_df.notna().sum().values,
    "missing_count": raw_df.isna().sum().values,
    "missing_pct": (raw_df.isna().mean().values * 100).round(2),
})

duplicate_row_count = raw_df.duplicated().sum()
print(f"Exact duplicate rows: {duplicate_row_count:,}")
display(column_profile)


In [ ]:
profile_output = CLEANED_DIR / "initial_column_profile.csv"
column_profile.to_csv(profile_output, index=False)
print(f"Saved column profile to: {profile_output.relative_to(PROJECT_ROOT)}")


## Profiling summary and next analytical step

The next notebook will clean and reshape the data after confirming the correct data sheet, area fields, year field, lower-quartile measures and row granularity.

Unusual values remain part of the review at this stage. The most unaffordable areas may look like outliers, but they are likely to be analytically important rather than errors.
